In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

CHOICES = ["A", "B", "C", "D"]
QUERY_TEMPLATE_MULTICHOICE = """
Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD. Think step by step before answering.

{Question}

A) {A}
B) {B}
C) {C}
D) {D}
""".strip()

ANSWER_PATTERN_MULTICHOICE = r"(?i)Answer[ \t]*:[ \t]*\$?([A-D])\$?"

In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048, # Choose any for long context!
    # load_in_4bit = False,  # 4 bit quantization to reduce memory
    # load_in_8bit = True, # [NEW!] A bit more accurate, uses 2x memory
    # full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

/data/common/ethanchang/ucct/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.54.0.dev0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.438 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [3]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [4]:
from datasets import load_dataset
## TEST             : 14K   ROWS
## AUXILIARY TRAIN  : 99.8K ROWS
## VALIDATION       : 1.53K ROWS
dataset = load_dataset("cais/mmlu", "all", split="validation")

In [5]:
# from unsloth.chat_templates import standardize_data_formats
# dataset = standardize_data_formats(dataset)

In [6]:
dataset[0]

{'question': 'The cyclic subgroup of Z_24 generated by 18 has order',
 'subject': 'abstract_algebra',
 'choices': ['4', '8', '12', '6'],
 'answer': 0}

In [7]:
# apply chat templates
def formatting_prompts_func(examples):
    query = QUERY_TEMPLATE_MULTICHOICE.format(Question=examples["question"], **{choice: text for choice, text in zip(CHOICES, examples["choices"])})  # examples["question"]
    text = tokenizer.apply_chat_template([{"role" : "user", "content": query}], tokenize = False, add_generation_prompt = True)
    return { "text" :  text }

dataset = dataset.map(formatting_prompts_func, batched = False)
dataset[100]

{'question': 'Reduction of D-xylose with NaBH4 yields a product that is a',
 'subject': 'college_chemistry',
 'choices': ['racemic mixture',
  'single pure enantiomer',
  'mixture of two diastereomers in equal amounts',
  'meso compound'],
 'answer': 3,
 'text': "<|im_start|>user\nAnswer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD. Think step by step before answering.\n\nReduction of D-xylose with NaBH4 yields a product that is a\n\nA) racemic mixture\nB) single pure enantiomer\nC) mixture of two diastereomers in equal amounts\nD) meso compound<|im_end|>\n<|im_start|>assistant\n"}

In [8]:
from transformers import TextStreamer
text = dataset[100]["text"]
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1000, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt=True)
)

Let's solve this step by step:

1) D-xylose is an aldopentose with multiple chiral centers. It has 3 chiral centers (C2, C3, C4).

2) NaBH4 is a reducing agent that reduces aldehyde groups to primary alcohols.

3) When D-xylose is reduced:
- The aldehyde group at C1 is reduced to a primary alcohol
- The stereochemistry at the existing chiral centers (C2, C3, C4) remains unchanged
- The reduction doesn't affect the chiral centers

4) The product is a single molecule with the same stereochemistry as D-xylose at C2, C3, and C4.

5) Since the reduction doesn't create new chiral centers or affect existing ones, and we're starting with a single enantiomer (D-xylose), we get a single pure enantiomer.

6) The reduction doesn't create a meso compound because:
- A meso compound requires identical chiral centers with opposite configurations
- D-xylose doesn't have this symmetry

7) The reaction doesn't produce a racemic mixture because:
- We're starting with a single enantiomer (D-xylose)
- The r